# AIPerf Use Case 3 — Trace-Based Benchmarking with the Mooncake Trace

Replay **real production traffic** at an endpoint instead of synthetic prompts, using the open-sourced Mooncake arXiv-Q&A trace and `--custom-dataset-type mooncake_trace`.

Based on the [AIPerf office-hours gist](https://gist.github.com/BenHamm/31c648f7d7331c94c1f3a45859db6677); notes in `docs/reference/aiperf-office-hours.md`.

## Table of contents

1. [Overview](#1-overview)
2. [Requirements](#2-requirements)
3. [Input — the Mooncake trace](#3-input--the-mooncake-trace)
4. [Test run](#4-test-run)
5. [Results analysis](#5-results-analysis)

## 1. Overview

Synthetic benchmarks (fixed or sampled ISL/OSL) are repeatable but unrealistic: every prompt is independent random tokens, so prefix/KV-cache reuse never happens. The Mooncake trace fixes that with a clever anonymization:

- Each record is a **timestamp + input/output lengths + block hash IDs** (each hash id stands for a fixed-size token block).
- **Repeated hash IDs across requests = shared prefixes** (e.g. follow-up questions about the same uploaded paper) — real reuse structure, zero user data.
- AIPerf expands the hashes into natural-language text **while preserving the repetition**, so a KV-cache-enabled server sees realistic cache hits.

Primary application: **A/B testing KV-cache optimizations** (e.g. Dynamo's KV-aware routing) — replay the same trace at two endpoints and compare TTFT. Synthetic prompts cannot show this difference at all.

> **Naming collision with this repo:** the Model Selection / Sizing scripts also pass `--custom-dataset-type mooncake_trace`, but only as a *file format* for deterministic fixed-order replay of our own prompt files (see `docs/scenarios/README.md`). This notebook uses the **actual Mooncake dataset with real timestamps** — a different thing.

## 2. Requirements

- `aiperf` CLI installed and on `PATH` (from the NGC AIPerf image, or `pip install aiperf`). The office-hours gist pinned `release/0.3.0`; pin whatever version you use (repo convention: record the AIPerf version per run).
- A reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...) serving the model under test.
- Notebook Python deps: `pip install -r notebooks/requirements.txt` (jupyter, pandas, matplotlib).
- Tip: AIPerf's live dashboard is designed for a terminal. If it renders poorly inside Jupyter, check `aiperf profile --help` for a plain/simple UI option in your version.
- Internet access to download the trace (~23,600 requests, one-time).
- **Context-window caution:** the trace's median ISL is ~6.4K tokens with a long tail past 100K. Requests longer than your endpoint's context limit will fail — the gist's demo lost 4% of requests to a 32K limit. Expect some errors unless you filter the trace or serve a long-context model.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)
print(f"Repo root: {REPO_ROOT}")

aiperf_path = shutil.which("aiperf")
if aiperf_path is None:
    print("aiperf CLI not found on PATH — install it before running the Test run section.")
else:
    print(f"aiperf found at: {aiperf_path}")
    version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
    print((version.stdout or version.stderr).strip())


## 3. Input — the Mooncake trace

Trace characteristics (per the gist): 23,608 requests over 60 minutes (~394 req/min), median ISL 6,402 tokens, mean 8,772, max 125,878. ~82% of requests are under 10K tokens.

The cells below download the trace, inspect it, and cut a **5-minute slice sped up 5×** (the gist's demo workload — replays in ~1 minute).

In [ ]:
import urllib.request

TRACE_URL = (
    "https://raw.githubusercontent.com/kvcache-ai/Mooncake/refs/heads/main/"
    "FAST25-release/arxiv-trace/mooncake_trace.jsonl"
)
trace_path = REPO_ROOT / "notebooks" / "data" / "mooncake_trace.jsonl"
trace_path.parent.mkdir(parents=True, exist_ok=True)

if not trace_path.exists():
    print(f"Downloading {TRACE_URL} ...")
    urllib.request.urlretrieve(TRACE_URL, trace_path)
print(f"Trace: {trace_path} ({trace_path.stat().st_size / 1e6:.1f} MB)")


In [ ]:
import pandas as pd

trace = pd.read_json(trace_path, lines=True)
print(f"{len(trace)} requests over {(trace.timestamp.max() - trace.timestamp.min()) / 60000:.1f} minutes")
print(trace.head(3).to_string())

# Prefix-reuse structure: how many hash blocks were already seen in an earlier request?
seen, reused, total = set(), 0, 0
for ids in trace["hash_ids"]:
    for h in ids:
        total += 1
        if h in seen:
            reused += 1
        seen.add(h)
print(f"\nBlock reuse: {reused}/{total} block references ({100 * reused / total:.0f}%) repeat an earlier block")
print("(this is the KV-cache-hit potential that synthetic prompts lack)")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
trace["input_length"].plot.hist(bins=60, ax=axes[0], log=True)
axes[0].set_title("ISL distribution (log count)")
axes[0].set_xlabel("input tokens")

(trace["timestamp"] // 60000).value_counts().sort_index().plot(ax=axes[1])
axes[1].set_title("Request rate over the hour")
axes[1].set_xlabel("minute")
axes[1].set_ylabel("requests/min")
plt.tight_layout()


In [ ]:
# Cut the gist's demo workload: first 5 minutes, timestamps compressed 5x -> ~1 minute replay.
SLICE_MINUTES = 5
SPEEDUP = 5

t0 = trace["timestamp"].min()
sliced = trace[trace["timestamp"] - t0 < SLICE_MINUTES * 60_000].copy()
sliced["timestamp"] = ((sliced["timestamp"] - t0) / SPEEDUP).astype(int)

slice_path = trace_path.with_name(f"mooncake_trace_{SLICE_MINUTES}min_{SPEEDUP}x.jsonl")
sliced.to_json(slice_path, orient="records", lines=True)
print(f"{len(sliced)} requests -> {slice_path.name} (replays in ~{SLICE_MINUTES * 60 // SPEEDUP}s)")


## 4. Test run

Two modes, per the gist:

- **With `--fixed-schedule`** — requests fire on the trace's timestamps: end-to-end system testing under the real arrival pattern (used below).
- **Without it** — the trace replays as fast as possible: capacity testing that still keeps the naturalistic prefix-reuse structure.

In [ ]:
def run_aiperf(cmd):
    """Print and run an aiperf command from the repo root, streaming output into the notebook."""
    print("$ " + " \\\n    ".join(cmd))
    result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
    print(f"\nexit code: {result.returncode}")
    return result

# ---- Required ----------------------------------------------------------
MODEL = ""       # model name the endpoint serves
URL = ""         # e.g. "http://localhost:8000"

TOKENIZER = ""   # empty = use MODEL
FIXED_SCHEDULE = True
# Relative to REPO_ROOT (aiperf runs with cwd=REPO_ROOT) — keep artifacts under notebooks/.
OUTPUT_DIR = "notebooks/artifacts/aiperf-uc3-mooncake"

assert MODEL and URL, "Set MODEL and URL above."

cmd = [
    "aiperf", "profile",
    "--model", MODEL,
    "--url", URL,
    "--endpoint-type", "chat",
    "--streaming",
    "--input-file", str(slice_path),
    "--custom-dataset-type", "mooncake_trace",
    "--tokenizer", TOKENIZER or MODEL,
    "--artifact-dir", OUTPUT_DIR,
]
if FIXED_SCHEDULE:
    cmd.append("--fixed-schedule")
run_aiperf(cmd)


## 5. Results analysis

Unlike the synthetic run, ISL/OSL now *vary per request*, so their distributions appear in the summary alongside the latency metrics. Gist reference numbers for this exact workload (5-min slice, 5× speed, overprovisioned endpoint): TTFT avg ≈ 407 ms, request throughput ≈ 26.7 req/s, 96% success (the 4% failures exceeded the 32K context limit).

In [ ]:
from io import StringIO

import pandas as pd

artifact_dir = REPO_ROOT / OUTPUT_DIR
csv_path = next(artifact_dir.rglob("profile_export_aiperf.csv"), None)
assert csv_path is not None, f"No profile_export_aiperf.csv under {artifact_dir} — did the Test run complete?"


def read_export(path):
    """Split the multi-section AIPerf CSV into (stats, totals) DataFrames.

    Section 1: per-request statistics (Metric, avg, min, max, p50, ..., std).
    Section 2: single-value run totals (Metric, Value).
    Section 3 (GPU telemetry, if present) is ignored here.
    """
    sections = Path(path).read_text().strip().split("\n\n")
    stats = pd.read_csv(StringIO(sections[0]))
    totals = (pd.read_csv(StringIO(sections[1]))
              if len(sections) > 1 else pd.DataFrame(columns=["Metric", "Value"]))
    return stats, totals


stats, totals = read_export(csv_path)
pd.set_option("display.max_rows", None)
display(stats)
totals


In [ ]:
# Latency and per-request length distributions (statistics section) — with a trace,
# ISL/OSL vary per request, so their distributions are informative here.
key = ["time to first token", "request latency", "inter token latency",
       "input sequence length", "output sequence length"]
mask = stats["Metric"].str.lower().apply(lambda n: any(p in n for p in key))
display(stats.loc[mask, ["Metric", "avg", "p50", "p90", "p99"]])

# Run totals: throughput, request count, duration, and any error counters.
totals


### Notes — A/B testing with a trace

- To measure a KV-cache optimization: run this identical command against endpoint A (optimization off) and endpoint B (on), and compare TTFT. With ~real prefix reuse, cache hits skip prefill work and TTFT drops; Dynamo's repo documents a smart-router A/B using exactly this method.
- Scale the load by editing the trace (e.g. duplicate every record 4× on the same schedule) — AIPerf replays whatever you give it.
- Your own saved production traces work the same way (see AIPerf's docs for accepted formats); ShareGPT is also supported.
- Errors from over-limit requests are reported in the export — check the error-rate rows before comparing latency numbers across runs.